# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-DAY02-NguyenVanXuanLoc-2A202602870"
CODE_DIR = REPO_DIR + '/submissions/2A202602870_NguyenVanXuanLoc/code'
sys.path.insert(0, CODE_DIR); sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # sau khi giải nén: mọi .jpg phải nằm trực tiếp ở đây
LABELS_DIR = "data/labels"
BASE = f"num_workers=4 images_dir={IMAGES_DIR} labels_dir={LABELS_DIR}"
os.chdir(CODE_DIR)           # các script dưới đây chạy trong thư mục code/ (data/ runs/ predictions/ ... là symlink hoặc thư mục cục bộ)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# Bước 0: EDA + kiểm tra split + kiểm tra pipeline (loss ban đầu ~ ln 9, overfit 1 batch, augmentation)
!python step0_checks.py --images {IMAGES_DIR} --labels {LABELS_DIR}
from IPython.display import Image, display
for f in ("step0_class_distribution", "step0_samples", "step0_augmentation"): display(Image(f"results/{f}.png"))

In [ ]:
# Kiểm tra đơn vị trên CPU (focal gamma=0 == CE, CutMix, gộp BN, temperature, ECE, overfit một batch)
!python -m unittest test_pipeline

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# Bước 1: 5 backbone (ResNet-50, ConvNeXt-T, ViT-S, MobileNetV3, EfficientNet-B0), cùng công thức T00, seed 0
!python run_all.py --stage B --epochs 5 --base {BASE}

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Bước 2: T00 và các trục augmentation / loss / sampler / EMA / cutmix, mỗi lần đổi một yếu tố
# (backbone đi tiếp = backbone tốt nhất ở Bước 1 theo val)
!python run_all.py --stage T --backbone convnext_tiny --epochs 5 --only T00 T01 T02 T03 T05 T07 T08 --base {BASE}

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# Bước 3: các phương pháp suy luận trên VAL + độ trễ p50/p95/p99 (warmup, synchronize, >=50 lần)
!python infer_all.py --run runs/F01/seed0 --b-glob "runs/B0*/seed0"
import pandas as pd; display(pd.read_csv("results/inference_val.csv"))

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# Bước 4: mốc T00 (test từ checkpoint hoặc cùng lúc huấn luyện, đúng một lần/seed) và chung kết F01
# F01 chỉ huấn luyện; test chạy đúng một lần/seed trong finalize.py với phương pháp suy luận chốt trên val.
!python run_all.py --stage final --backbone convnext_tiny --epochs 5 --over loss=ls --base {BASE}
!python finalize.py --exp F01 --views flip --space prob --temperature --base {BASE}

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# Bước 5: gom kết quả thật thành results.xlsx (7 sheet)
!python make_results.py